# OncoBridge AI — Generar imágenes de referencia (Stable Diffusion, GPU)

Este notebook genera las 30 imágenes de referencia del ground truth (una por `gt_id`), usando el mismo modelo y los mismos prompts que usa `image_synthesizer.py` del proyecto (`Nihirc/Prompt2MedImage`, Stable Diffusion afinado con imágenes médicas reales del dataset ROCO).

**Antes de correr esto:**
1. En tu compu, andá a `data/dataset_clinical_only/dataset/oncology_ground_truth_base/`.
2. Seleccioná los 30 archivos `.json` (o toda la carpeta), click derecho → **Enviar a → Carpeta comprimida (zip)**.
3. Antes de correr las celdas: arriba en el menú, andá a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** y guardá.

Corré las celdas en orden, de arriba hacia abajo (▶ en cada una, o Ctrl+Enter).

In [ ]:
!pip install -q diffusers transformers accelerate torch pillow

## Subir el .zip con los 30 JSON del ground truth
Al correr la celda de abajo va a aparecer un botón "Choose Files" — elegí el .zip que armaste en el paso 2 de arriba.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import glob
import os
import zipfile

zip_name = list(uploaded.keys())[0]
os.makedirs("gt_data", exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("gt_data")

# Busca los .json en cualquier subcarpeta, por si el zip trae la carpeta adentro.
gt_files = sorted(glob.glob("gt_data/**/*.json", recursive=True))
print(f"Encontrados {len(gt_files)} archivos JSON")
assert len(gt_files) == 30, "Se esperaban 30 archivos GT -- revisá el zip subido"

## Cargar Stable Diffusion en GPU

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_NAME = "Nihirc/Prompt2MedImage"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
pipe = pipe.to("cuda")
print("Modelo cargado en GPU.")

## Generar las 30 imágenes de referencia
Usa `meddiffusion_prompt` / `meddiffusion_negative_prompt` de cada entrada GT -- los mismos campos que usa `image_synthesizer.py` en el proyecto.

In [ ]:
import json

os.makedirs("reference_images", exist_ok=True)

for i, path in enumerate(gt_files, start=1):
    with open(path, "r", encoding="utf-8") as f:
        entry = json.load(f)

    gt_id = entry["gt_id"]
    guidance = entry["radiologist_guidance"]
    prompt = guidance["meddiffusion_prompt"]
    negative_prompt = guidance["meddiffusion_negative_prompt"]

    out_path = f"reference_images/{gt_id}.png"
    if os.path.exists(out_path):
        print(f"[{i}/{len(gt_files)}] {gt_id} ya generado, salteo")
        continue

    image = pipe(prompt=prompt, negative_prompt=negative_prompt).images[0]
    image.save(out_path)
    print(f"[{i}/{len(gt_files)}] {gt_id} generado -> {out_path}")

print("\nListo, las 30 imagenes generadas.")

## Descargar el resultado
Arma un .zip con las 30 imágenes y lo descarga a tu compu.

In [ ]:
import shutil

shutil.make_archive("reference_images", "zip", "reference_images")
files.download("reference_images.zip")

## Último paso (en tu compu, no en Colab)

1. Descomprimí el `reference_images.zip` que se descargó.
2. Copiá los 30 `.png` (`GT-XXX.png`) adentro de `data/generated/reference_images/` en la carpeta del proyecto (creá la carpeta si no existe).
3. Corré de nuevo `python scripts/generate_reference_images.py` en tu terminal local -- como ya están todas cacheadas, va a decir "ya existe" para las 30 sin generar nada, confirmando que quedó todo bien.